In [1]:
%matplotlib inline
# Cell 1 — WO-02 Step 3, family 6: Categorical. Load fixtures and population payloads (read-only; the API is never called)
import sys
from pathlib import Path
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts' / 'edop').exists())
sys.path.insert(0, str(ROOT))
from scripts.edop.dashboard import wo02_payloads as P, wo02_rules_categorical as K, wo02_analysis as A

fx, pop, prov = P.load_all()
W = P.population_weights(pop)
FX_BY_ID = {q['id']: q for q in fx}
print('provenance:', prov)
print(f'fixtures with a 200: {len(fx)} | population queries with a 200: {len(pop)}')


provenance: {'app_version': '0.4', 'git_commit': 'bb80fee', 'git_describe': 'v0.4.2-12-gbb80fee', 'branch': 'dashboard', 'planned': '2026-10-30', 'captured': '2026-10-30'}
fixtures with a 200: 33 | population queries with a 200: 505


In [2]:
# Cell 2 — The four Categorical statement types (rules in scripts/edop/dashboard/wo02_rules_categorical.py)
for s in K.STATEMENTS:
    print(f"{s['id']}  {s['name']}")
    print(f"      applies to: {s['applies_to']}")
    print(f"      template  : {s['template']}")
print()
print(f"classes listed in CAT-03: {K.TOP_N} (a display choice; the residual after the top n is swept in Cell 7)")
print('mixture shares are normalised here by their own sum (the engine does not normalise outlet_type)')


CAT-01  The class (one basin)
      applies to: single basin and one-basin buffer; class_mixture (10 variables); L6, L8
      template  : {name}: {label}.
CAT-02  The most common class and its share (area)
      applies to: buffer, polity; class_mixture; L6, L8
      template  : {name}: {label} is the most common class, covering {share} of the area with a recorded class. | {name}: all of the area with a recorded class is {label}.
CAT-03  The mixture: top classes and the rest (area)
      applies to: buffer, polity; class_mixture; L6, L8
      template  : {name}: {label} {share}; {label} {share}; {label} {share}; {k} other classes {share}.
CAT-04  Concentrated or mixed (the engine's label) as wording
      applies to: buffer, polity; class_mixture; L6, L8
      template  : {name}: the classes are {concentrated|mixed}.

classes listed in CAT-03: 3 (a display choice; the residual after the top n is swept in Cell 7)
mixture shares are normalised here by their own sum (the engine does not n

In [3]:
# Cell 3 — Fixtures: apply every rule to every categorical row; count where each applies and why not
fdf = A.apply_fixtures(fx, K)
print(A.fixture_table(fdf).to_string())
print()
print('why a statement did not apply (count of rows):')
print(A.reason_table(fdf).to_string())


res                  applied  not_applicable
id     scope                                
CAT-01 buffer              0              50
       polity              0              40
       single_basin      230               0
CAT-02 buffer             50               0
       polity             40               0
       single_basin        0             230
CAT-03 buffer             30              20
       polity             40               0
       single_basin        0             230
CAT-04 buffer             30              20
       polity             40               0
       single_basin        0             230

why a statement did not apply (count of rows):
id      reason    
CAT-01  multi_unit     90
CAT-02  one_unit      230
CAT-03  one_class      20
        one_unit      230
CAT-04  one_class      20
        one_unit      230


In [4]:
# Cell 4 — Draft sentences on chosen fixtures (up to 3 per statement type), to read and judge
for qid in ['F01', 'F04', 'F18', 'F19', 'F22']:
    g = fdf[(fdf.qid == qid) & fdf.applies]
    print(f"--- {qid}: {FX_BY_ID[qid]['condition']}")
    for sid, gg in g.groupby('id'):
        for _, r in gg.head(3 if sid == 'CAT-03' else 2).iterrows():
            print(f"  {sid}  {r['text']}")
    print()


--- F01: arid; exogenous water (Niger); outside_active_domain
  CAT-01  Biome name: Flooded Grasslands & Savannas.
  CAT-01  Climate zone name: Extremely hot and xeric.

--- F04: large buffer; suppressed scores; section 2.6 reference counts
  CAT-02  Biome name: Tropical & Subtropical Grasslands, Savannas & Shrublands is the most common class, covering 71% of the area with a recorded class.
  CAT-02  Climate zone name: all of the area with a recorded class is Extremely hot and xeric.
  CAT-03  Biome name: Tropical & Subtropical Grasslands, Savannas & Shrublands 71%; Flooded Grasslands & Savannas 29%.
  CAT-03  Terrestrial ecoregion ID: Sahelian Acacia savanna 71%; Inner Niger Delta flooded savanna 29%.
  CAT-03  Freshwater ecoregion name: Inner Niger Delta 47%; Dry Sahel 44%; Lower Niger - Benue 9%.
  CAT-04  Biome name: the classes are mixed.
  CAT-04  Terrestrial ecoregion ID: the classes are mixed.

--- F18: low_coverage by sea (about 50% sea)
  CAT-02  Biome name: Mediterranean For

In [5]:
# Cell 5 — Population sample: how often each statement applies (share of categorical rows), weighted and unweighted
pdf = A.apply_population(pop, W, K)
print('categorical rows evaluated per statement type:', len(pdf) // len(K.STATEMENTS), '(10 variables per query)')
print('WEIGHTED by climate-zone size (percent of rows):')
print((A.rates(pdf) * 100).round(1).to_string())
print()
print('UNWEIGHTED (percent of rows):')
print((A.unweighted_rates(pdf) * 100).round(1).to_string())
print()
print('polities are not in the population sample (5 queries, all fixtures); see Cell 3')


categorical rows evaluated per statement type: 5050 (10 variables per query)
WEIGHTED by climate-zone size (percent of rows):
kind    buffer L6  single L6  single L8
id                                     
CAT-01        0.3       98.6       96.0
CAT-02       99.7        0.0        0.0
CAT-03       60.3        0.0        0.0
CAT-04       60.3        0.0        0.0

UNWEIGHTED (percent of rows):
kind    buffer L6  single L6  single L8
id                                     
CAT-01        1.0       97.3       94.8
CAT-02       98.4        0.0        0.0
CAT-03       55.8        0.0        0.0
CAT-04       55.8        0.0        0.0

polities are not in the population sample (5 queries, all fixtures); see Cell 3


In [6]:
# Cell 6 — The numbers inside the statements (weighted, buffer L6)
print('CAT-02 share of the most common class; CAT-03 share of the classes beyond the top 3; CAT-04 modal share:')
print(A.stat_quantiles(pdf, ['CAT-02', 'CAT-03', 'CAT-04']).round(3).to_string(index=False))
lab = pdf[pdf.applies & (pdf.id == 'CAT-04')]
num = lab.groupby(['kind', 'coherence']).w.sum()
den = lab.groupby('kind').w.sum()
print()
print('weighted share of multi-basin rows by the engine label (percent):')
print((num / num.index.get_level_values('kind').map(den).values * 100).round(1).to_string())
c, ncl = K.mixture_checks(pop + fx)
print()
print('number of classes in multi-basin rows (capped at 6), count:', dict(sorted(ncl.items())))
by = pdf[(pdf.id == 'CAT-02') & pdf.applies].groupby('variable').stat.apply(lambda s: float((s.astype(float) >= 0.999).mean())).round(2)
print('share of multi-basin rows with one class only, by variable:')
print(by.to_string())


CAT-02 share of the most common class; CAT-03 share of the classes beyond the top 3; CAT-04 modal share:
    id      kind   n   p10   p25   p50   p75   p90
CAT-02 buffer L6 974 0.584 0.725 0.939 1.000 1.000
CAT-03 buffer L6 552 0.000 0.000 0.000 0.000 0.004
CAT-04 buffer L6 552 0.535 0.629 0.773 0.897 0.973

weighted share of multi-basin rows by the engine label (percent):
kind       coherence   
buffer L6  concentrated    35.3
           mixed           64.7

number of classes in multi-basin rows (capped at 6), count: {1: 452, 2: 368, 3: 157, 4: 54, 5: 14, 6: 29}
share of multi-basin rows with one class only, by variable:
variable
biome                         0.53
eco_id                        0.30
freshwater_ecoregion_class    0.73
freshwater_ecoregion_name     0.52
land_cover_name               0.44
lith_class                    0.27
outlet_type                   0.39
pnv_majority                  0.51
wetland_class                 0.38
zone_name                     0.27


In [7]:
# Cell 7 — Sweeps: (a) the concentrated or mixed label (engine cutoff: modal share 0.85); (b) how many classes to list
CUTS = [0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95]
sw = A.cutoff_sweep(pdf, 'CAT-04', CUTS)
print('(a) share of multi-basin rows that would read "concentrated" at each modal-share cutoff, and share whose wording changes between 0.6 and 0.95:')
print(sw.round(3).to_string(index=False))
print()
print('variables whose label flips (count of rows with a modal share from 0.6 up to 0.95):')
print(A.flipping_variables(pdf, 'CAT-04', CUTS[0], CUTS[-1]).to_string())
print()
NS = [1, 2, 3, 4, 5]
res = K.residual_by_n(pop, NS)
r = pd.DataFrame([(q, v, k, *rs) for q, v, k, rs in res], columns=['qid', 'variable', 'k', *[f'n{n}' for n in NS]])
r['w'] = r.qid.map(W)
print('(b) weighted share of multi-basin rows where classes beyond the top n carry at least 1% of the area (percent):')
print({f'top {n}': round(float((r.w * (r[f'n{n}'] >= 0.01)).sum() / r.w.sum() * 100), 1) for n in NS})
print('    ... at least 5%:', {f'top {n}': round(float((r.w * (r[f'n{n}'] >= 0.05)).sum() / r.w.sum() * 100), 1) for n in NS})


(a) share of multi-basin rows that would read "concentrated" at each modal-share cutoff, and share whose wording changes between 0.6 and 0.95:
     kind   n  >= 0.6  >= 0.7  >= 0.75  >= 0.8  >= 0.85  >= 0.9  >= 0.95  flips
buffer L6 552   0.804   0.628    0.546   0.431    0.354   0.247    0.144   0.66

variables whose label flips (count of rows with a modal share from 0.6 up to 0.95):
variable
lith_class                   53
zone_name                    47
eco_id                       44
outlet_type                  43
wetland_class                36
freshwater_ecoregion_name    36
biome                        31
land_cover_name              27

(b) weighted share of multi-basin rows where classes beyond the top n carry at least 1% of the area (percent):
{'top 1': 57.2, 'top 2': 19.2, 'top 3': 5.2, 'top 4': 0.5, 'top 5': 0.0}
    ... at least 5%: {'top 1': 51.7, 'top 2': 15.0, 'top 3': 2.7, 'top 4': 0.2, 'top 5': 0.0}


In [8]:
# Cell 8 — Checks on the mixtures
c, ncl = K.mixture_checks(pop + fx)
for k, v in c.items():
    print(f'  {k}: {v}')
print()
print('engine label versus the label implied by the normalised modal share (rows that disagree):', dict(K.label_vs_normalised(pop + fx)))
print()
nd = fdf[(fdf.id == 'CAT-01') & fdf.applies]
print('classes literally labelled N/A: variable and rows (fixtures + population):', {k: v for k, v in c.items() if 'N/A' in k})


  rows: 5228
  weights sum to 1: 5191
  concentration = sum of squared shares: 5228
  sorted largest first: 5228
  an 'N/A' class (biome): 97
  weights do not sum to 1 (outlet_type): 37

engine label versus the label implied by the normalised modal share (rows that disagree): {('outlet_type', 'mixed', 'concentrated'): 2}

classes literally labelled N/A: variable and rows (fixtures + population): {"an 'N/A' class (biome)": 97}


In [9]:
# Cell 9 — Write this family's rows to the master table (replaces any earlier Categorical rows)
fxt, rw, ru = A.fixture_table(fdf), A.rates(pdf), A.unweighted_rates(pdf)
s4 = sw.iloc[0]
rows = []
for s in K.STATEMENTS:
    sid = s['id']
    f = fxt.loc[sid]
    row = dict(s, family='Categorical')
    row['fixture_result'] = f"{int(f.applied.sum())} rows applied, {int(f.not_applicable.sum())} not applicable ({len(fx)} fixtures)"
    row['population_result'] = ('share of categorical rows, weighted: ' + ', '.join(f"{k} {rw.loc[sid, k]:.1%}" for k in rw.columns)
                                + '; unweighted: ' + ', '.join(f"{k} {ru.loc[sid, k]:.1%}" for k in ru.columns) + '; polities not sampled')
    if sid == 'CAT-04':
        row['sweep_result'] = (f"label flips: share reading 'concentrated' is {s4['>= 0.6']:.0%} at a modal-share cutoff of 0.6 and {s4['>= 0.95']:.0%} at 0.95; "
                               f"{s4['flips']:.0%} of rows change wording between 0.6 and 0.95; CAT-02 states the share")
    elif sid == 'CAT-03':
        row['sweep_result'] = 'how many classes to list is a display choice; the residual after the top n is reported (Cell 7)'
    else:
        row['sweep_result'] = 'no cutoff'
    rows.append(row)
n_rows = P.upsert_master('Categorical', rows)
print('master table now has', n_rows, 'rows ->', P.MASTER)


master table now has 32 rows -> /Users/karlg/Documents/repos/_edops/notebooks/edop/dashboard/wo02_master_table.csv
